# 00 · Setup and NIM health

**Purpose** — bring the three NIM microservices up on this GPU box and prove they are reachable.

**Prerequisites** — an `nvapi-...` key with NGC catalog access, exported as `NVIDIA_API_KEY`
or placed in `.env` at the repo root. A GPU with >=40GB free (L40S 48GB is the target).

> The key is never printed by this notebook. It is read from the environment and
> only a masked suffix is shown.

In [ ]:
import sys, os, json
sys.path.insert(0, os.path.abspath(".."))          # notebooks/ -> repo root
os.chdir(os.path.abspath(".."))
from dotenv import load_dotenv; load_dotenv()
print("repo root:", os.getcwd())

## Config — key check (masked)

In [ ]:
from getpass import getpass
key = os.environ.get("NVIDIA_API_KEY") or os.environ.get("NGC_API_KEY")
if not key:
    key = getpass("NVIDIA API key (nvapi-...): ")
    os.environ["NVIDIA_API_KEY"] = key
os.environ.setdefault("NGC_API_KEY", key)
assert key.startswith("nvapi-"), "key should start with nvapi-"
print("key loaded:  nvapi-...%s  (len %d)" % (key[-4:], len(key)))

## Execute — GPU check

If this reports less than ~40GB free, use the 1B reranker instead of the 4B one
(see `scripts/start_nims.sh`, `RRK_IMG`).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.used,memory.free --format=csv

## Execute — pull the NIM containers

Long pole: 30–90 minutes including TensorRT engine builds. Run this **first** and
work through notebooks 01 and 02 (neither needs a GPU) while it downloads.

In [ ]:
!bash scripts/start_nims.sh pull

## Execute — start the three services

In [ ]:
!bash scripts/start_nims.sh run

## What you should see

All three reporting `OK`, and VRAM around 39–41GB used of 48GB.
First start builds TRT engines, so allow 10–20 minutes before they turn healthy.
Watch one with `!bash scripts/start_nims.sh logs llm`.

In [ ]:
!bash scripts/start_nims.sh health

In [ ]:
# Round-trip test against each endpoint.
import httpx
for name, port, path in [("llm", 8000, "/v1/models"),
                         ("embed", 8001, "/v1/models"),
                         ("rerank", 8002, "/v1/models")]:
    try:
        r = httpx.get(f"http://localhost:{port}{path}", timeout=5)
        ids = [m.get("id") for m in r.json().get("data", [])]
        print(f"  OK      {name:7s} :{port}  -> {ids}")
    except Exception as e:
        print(f"  NOT UP  {name:7s} :{port}  ({type(e).__name__})")